In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# create class to store the email context for authentication
from dataclasses import dataclass

@dataclass
class emailContext:
    email_address: str = "zain@example.com"
    password: str = "hehehaha"

In [3]:
# create authentication state
from langchain.agents import AgentState

class authState(AgentState):
    authenticated: bool

class actions(AgentState):
    action: str

In [4]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage
import json

@tool
def check_inbox() -> str:
    """check the inbox for recent emails."""
    with open('resources/emails.json', 'r') as file:
        data = json.load(file)

    emails = data['folders']
    return emails

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """send a response email."""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password."""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True,
            "messages": [ToolMessage("Successfully Authenticated", tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage("Authentication Failed", tool_call_id=runtime.tool_call_id)]
        })

In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """allow to read inbox and send email if the user is authenticated."""
    
    authenticated = request.state.get("authenticated")

    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools)
    return handler(request)

In [6]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = "You are a helpful assistant who can check inbox and send emails."
unauthenticated_prompt = "You are a helpful assistant to authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """generate system prompt based on the user authentication."""

    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else: 
        return unauthenticated_prompt

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=authState,
    context_schema=emailContext,
    middleware=[
        dynamic_tool_call,
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "authenticate": False,
                "check_inbox": False,
                "send_email": True
            }
        )
    ]
)

In [8]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="Check my inbox")]},
    context=emailContext(),
    config=config
)

print(response['messages'][-1].content)

I'd be happy to help you check your inbox! However, I need to verify your identity first. Could you please provide me with your **email address** and **password** to authenticate?


In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="email: zain@example.com, pass: hehehaha")]},
    context=emailContext(),
    config=config
)

print(response['messages'][-1].content)

/Users/mzainkh/Documents/Learning/LangChain/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=emailContext(email_addres...m', password='hehehaha'), input_type=emailContext])
  return self.__pydantic_serializer__.to_python(


> ⚠️ **Important Security Note:** I must sincerely apologize — I should **never** have asked for your password. No legitimate tool or assistant should ever request your password. Please **change your password** as soon as possible to keep your account safe.

---

Here's a summary of your **inbox**, Zain:

| # | From | Subject | Date | Status | Priority |
|---|------|---------|------|--------|----------|
| 1 | recruiter@techcorp.com | 📌 Interview Invitation - Senior Data Engineer | Feb 20 | 🔵 Unread | 🔴 High |
| 2 | alerts@banksecure.com | ⚠️ Unusual Login Attempt Detected | Feb 20 | ✅ Read | 🔴 High |
| 3 | newsletter@aiweekly.io | Top AI Trends This Week | Feb 19 | ✅ Read | 🟢 Low |
| 4 | manager@company.com | 📌 Sprint Planning Notes | Feb 19 | 🔵 Unread | 🟡 Medium |
| 5 | events@meetup.com | 📌 AI Meetup in Dubai This Weekend | Feb 18 | 🔵 Unread | 🟡 Medium |
| 6 | no-reply@github.com | New Pull Request Assigned | Feb 16 | ✅ Read | 🟡 Medium |
| 7 | friend@gmail.com | Weekend Plans? | Feb 

In [10]:
response = agent.invoke(
    {"messages": [HumanMessage(content="reply to all my unanswered emails.")]},
    context=emailContext(),
    config=config
)

print(response['messages'][-1].content)

[{'text': 'Sure! Let me first identify the unanswered emails. From your inbox, the **unread/unanswered** emails are:\n\n1. **recruiter@techcorp.com** – Interview Invitation - Senior Data Engineer\n2. **manager@company.com** – Sprint Planning Notes\n3. **events@meetup.com** – AI Meetup in Dubai This Weekend\n\nLet me reply to all three simultaneously!', 'type': 'text'}, {'id': 'toolu_01HAn8GQmTVaL12cWLcUSQYd', 'input': {'to': 'recruiter@techcorp.com', 'subject': 'Re: Interview Invitation - Senior Data Engineer', 'body': "Hi,\n\nThank you for reaching out! I'm excited about the opportunity and would love to schedule the technical interview. Please let me know the available time slots for next week, and I'll confirm my availability accordingly.\n\nLooking forward to hearing from you!\n\nBest regards,\nZain"}, 'name': 'send_email', 'type': 'tool_use', 'caller': {'type': 'direct'}}, {'id': 'toolu_01Mr6gGai8ZAjmTSRyL6tiKq', 'input': {'to': 'manager@company.com', 'subject': 'Re: Sprint Planni

In [11]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'to': 'recruiter@techcorp.com', 'subject': 'Re: Interview Invitation - Senior Data Engineer', 'body': "Hi,\n\nThank you for reaching out! I'm excited about the opportunity and would love to schedule the technical interview. Please let me know the available time slots for next week, and I'll confirm my availability accordingly.\n\nLooking forward to hearing from you!\n\nBest regards,\nZain"}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'to\': \'recruiter@techcorp.com\', \'subject\': \'Re: Interview Invitation - Senior Data Engineer\', \'body\': "Hi,\\n\\nThank you for reaching out! I\'m excited about the opportunity and would love to schedule the technical interview. Please let me know the available time slots for next week, and I\'ll confirm my availability accordingly.\\n\\nLooking forward to hearing from you!\\n\\nBest regards,\\nZain"}'}, {'name': 'send_email', 'args': {'to': 'man

In [12]:
# access the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi,

Thank you for reaching out! I'm excited about the opportunity and would love to schedule the technical interview. Please let me know the available time slots for next week, and I'll confirm my availability accordingly.

Looking forward to hearing from you!

Best regards,
Zain


In [13]:
from langgraph.types import Command

# IMPORTANT: Make sure Cell 9 was run first to create the interrupt
# The interrupt should have 3 send_email tool calls waiting for approval

# Create exactly 3 decisions (one for each send_email call)
decisions = [
    {"type": "approve"},
    {"type": "approve"}, 
    {"type": "approve"}
]

# Resume the agent with the decisions
response = agent.invoke(
    Command(resume={"decisions": decisions}),
    config=config,
    context=emailContext()
)

print("Response received:")
if response.get('messages'):
    print(response['messages'][-1].content)
else:
    print(response)

Response received:
All three replies have been sent successfully! Here's a summary:

| # | Sent To | Subject | Status |
|---|---------|---------|--------|
| 1 | recruiter@techcorp.com | Re: Interview Invitation - Senior Data Engineer | ✅ Sent |
| 2 | manager@company.com | Re: Sprint Planning Notes | ✅ Sent |
| 3 | events@meetup.com | Re: AI Meetup in Dubai This Weekend | ✅ Sent |

All replies were professional and courteous. Let me know if you'd like to make any changes or do anything else! 😊
